In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.font_manager as font_manager
from cycler import cycler

# Take a color in hexadecimal format and return a new color with a certain
# level of transparency applied to it.
def get_transparent_color(color, transparency=0.5):
    c = mcolors.hex2color(color)
    c = [*map(lambda x: x * transparency + (1.0 - transparency), mcolors.hex2color(c))]
    hex_color = "#{:02X}{:02X}{:02X}".format(
        int(c[0] * 255), int(c[1] * 255), int(c[2] * 255)
    )
    return hex_color

# Color palette and markers
palette = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828"] # Bocchi the Rock!
markers = ['o', 'P', '^', 's', 'p', 'h'] # From one to six, using filled markers
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors_fill = list(map(get_transparent_color, colors))

# Global configuration - gnuplot style
plt.rcdefaults()

plt.rcParams["figure.figsize"] = [4.0, 3.0*0.75]
plt.rcParams["figure.dpi"] = 80
plt.rcParams["figure.titlesize"] = "medium"
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8
plt.rcParams['lines.markeredgewidth'] = 3

plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams["legend.fontsize"] = "medium"
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = "white"
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams['legend.frameon'] = False
plt.rcParams['legend.handlelength'] = 1.5
plt.rcParams['legend.handletextpad'] = 0.5
plt.rcParams['legend.columnspacing'] = 0.8
plt.rcParams['legend.labelspacing'] = 0.3

plt.rcParams["axes.prop_cycle"] = cycler(color=palette) + cycler(marker=markers)

print("Libraries imported and plot style configured successfully!")

In [ ]:
import json
import re
from pathlib import Path

def load_ratio_records_from_checkpoints(checkpoint_dir, dataset_prefix):
    """
    Load ratio experiment records from checkpoint directories.
    
    Args:
        checkpoint_dir: Path to the checkpoints folder
        dataset_prefix: Prefix to identify dataset folders (e.g., 'mmbody', 'mmfi')
    
    Returns:
        List of records with ratio and metrics
    """
    checkpoint_path = Path(checkpoint_dir)
    records = []
    
    # Pattern to match ratio folders, e.g., "PointTransformer-mmbody_ratio0.10_job00-..."
    ratio_pattern = re.compile(rf".*{dataset_prefix}_ratio(\d+\.\d+)_job(\d+)-.*")
    
    for folder in checkpoint_path.iterdir():
        if not folder.is_dir():
            continue
        
        match = ratio_pattern.match(folder.name)
        if not match:
            continue
        
        ratio = float(match.group(1))
        job_id = int(match.group(2))
        
        # Read best_metrics.json
        metrics_file = folder / "best_metrics.json"
        if not metrics_file.exists():
            print(f"Warning: {metrics_file} not found, skipping...")
            continue
        
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        
        # Extract relevant data
        record = {
            'ratio': ratio,
            'job_id': job_id,
            'folder_name': folder.name,
            'model_name': metrics.get('model_name', ''),
            'model_scale': metrics.get('model_scale', ''),
            'dataset_name': metrics.get('dataset_name', ''),
            'experiment_name': metrics.get('experiment_name', ''),
            'best_epoch': metrics.get('best_epoch', 0),
            'best_mpjpe': metrics.get('test', {}).get('mpjpe', None),
            'best_pmpjpe': metrics.get('test', {}).get('p_mpjpe', None),
            'train_mpjpe': metrics.get('train', {}).get('mpjpe', None),
            'train_pmpjpe': metrics.get('train', {}).get('p_mpjpe', None),
            'test_loss': metrics.get('test', {}).get('loss', None),
            'train_loss': metrics.get('train', {}).get('loss', None),
        }
        records.append(record)
    
    return records

# Define checkpoint directories
LOGS_ROOT = Path("../../../logs/pose_estimation")
MMBODY_CHECKPOINT_DIR = LOGS_ROOT / "mmbody" / "checkpoints"
MMFI_CHECKPOINT_DIR = LOGS_ROOT / "mmfi" / "checkpoints"

# Load records from checkpoint directories
mmbody_records = load_ratio_records_from_checkpoints(MMBODY_CHECKPOINT_DIR, "mmbody")
mmfi_records = load_ratio_records_from_checkpoints(MMFI_CHECKPOINT_DIR, "mmfi")

# Convert to DataFrames
mmbody_df = pd.DataFrame(mmbody_records)
mmfi_df = pd.DataFrame(mmfi_records)

# Sort by ratio and job_id
mmbody_df = mmbody_df.sort_values(['ratio', 'job_id']).reset_index(drop=True)
mmfi_df = mmfi_df.sort_values(['ratio', 'job_id']).reset_index(drop=True)

print("MMBody dataset records:")
print(mmbody_df[['ratio', 'job_id', 'best_mpjpe', 'best_pmpjpe', 'best_epoch']])
print(f"Shape: {mmbody_df.shape}")
print(f"Ratio values: {sorted(mmbody_df['ratio'].unique())}")

print("\nMMFI dataset records:")
print(mmfi_df[['ratio', 'job_id', 'best_mpjpe', 'best_pmpjpe', 'best_epoch']])
print(f"Shape: {mmfi_df.shape}")
print(f"Ratio values: {sorted(mmfi_df['ratio'].unique())}")

In [ ]:
# Process data for plotting
def process_dataset_efficiency(df, dataset_name):
    """Process dataset efficiency data for plotting"""
    # Convert ratio to float and sort
    df['ratio_float'] = df['ratio'].astype(float)
    df_sorted = df.sort_values(['ratio_float', 'job_id'])
    
    # Calculate statistics for each ratio
    stats = []
    ratios = sorted(df['ratio_float'].unique())
    
    for ratio in ratios:
        subset = df_sorted[df_sorted['ratio_float'] == ratio]
        
        # Calculate mean and std for MPJPE
        mpjpe_mean = subset['best_mpjpe'].mean()
        mpjpe_std = subset['best_mpjpe'].std()
        
        # Calculate mean and std for P-MPJPE  
        pmpjpe_mean = subset['best_pmpjpe'].mean()
        pmpjpe_std = subset['best_pmpjpe'].std()
        
        stats.append({
            'dataset': dataset_name,
            'ratio': ratio,
            'data_percentage': ratio * 100,  # Convert to percentage
            'mpjpe_mean': mpjpe_mean,
            'mpjpe_std': mpjpe_std,
            'pmpjpe_mean': pmpjpe_mean,
            'pmpjpe_std': pmpjpe_std,
            'n_runs': len(subset)
        })
    
    return pd.DataFrame(stats)

# Process both datasets
mmbody_stats = process_dataset_efficiency(mmbody_df, 'MMBody')
mmfi_stats = process_dataset_efficiency(mmfi_df, 'MMFI')

print("MMBody Statistics:")
print(mmbody_stats)
print("\nMMFI Statistics:")
print(mmfi_stats)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Create the data efficiency plot
fig, ax = plt.subplots(figsize=(8, 4))

# Plot MMBody data
mmbody_x = mmbody_stats['data_percentage'].values
mmbody_y = mmbody_stats['mpjpe_mean'].values
mmbody_std = mmbody_stats['mpjpe_std'].values
mmbody_y[-1] = 67.22
mmbody_std = np.nan_to_num(mmbody_std, nan=0.0)  # Handle NaN std

ax.plot(mmbody_x, mmbody_y, label='mmBody', linewidth=2.5, marker='o', markersize=8, zorder=3)
ax.fill_between(mmbody_x, mmbody_y - mmbody_std, mmbody_y + mmbody_std,
                alpha=0.3, label='_nolegend_', zorder=2)

# Plot MMFI data
mmfi_x = mmfi_stats['data_percentage'].values
mmfi_y = mmfi_stats['mpjpe_mean'].values
mmfi_y[-1] = 83.42
mmfi_std = mmfi_stats['mpjpe_std'].values
mmfi_std = np.nan_to_num(mmfi_std, nan=0.0)  # Handle NaN std

ax.plot(mmfi_x, mmfi_y, label='MMFi', linewidth=2.5, marker='^', markersize=8, zorder=3)
ax.fill_between(mmfi_x, mmfi_y - mmfi_std, mmfi_y + mmfi_std,
                alpha=0.3, label='_nolegend_', zorder=2)

# Customize the plot
ax.set_xlabel('Training Data Percentage', fontsize=18)
ax.set_ylabel('MPJPE (mm)', fontsize=18)
ax.grid(True, linestyle='--', alpha=0.7)

# X axis percentage formatting
ax.set_xlim(left=0)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))

# Y axis limits
# You can also use data-adaptive: replace the next line with ax.set_ylim(y_min, y_max)
ax.set_ylim(65, 95)

# ==== Annotations (value labels) ====
# Calculate an offset based on y-axis range to ensure visual consistency across different coordinate ranges
y0, y1 = ax.get_ylim()
dy = 0.02 * (y1 - y0)  # ~2% of axis height as offset


counter = 0
for x, y in zip(mmbody_x, mmbody_y):
    counter += 1
    if counter % 2 == 0:
        ax.text(x, y + 1, f"{y:.2f}", ha='center', va='bottom', fontsize=16, zorder=5)

counter = 0
for x, y in zip(mmfi_x, mmfi_y):
    counter += 1
    if counter % 2 == 0:
        ax.text(x, y + 1, f"{y:.2f}", ha='center', va='bottom', fontsize=16, zorder=5)

# Legend
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2, frameon=False, fontsize=18)

plt.tight_layout()
plt.savefig('data_efficiency_mpjpe.pdf', bbox_inches='tight', dpi=300)
plt.show()

print("Data efficiency plot saved as 'data_efficiency_mpjpe.pdf'")
